# Zero-shot VQA on RadImageNet-VQA: Gemma 3 4B-it
Closed-ended (yes/no) and multiple-choice questions only. 500 questions, 5 groups x 100, seed 42.
Same prompts, sampling and scoring as the other three notebooks; only the model cells differ.

## 1. Setup and Hugging Face login

In [1]:
!pip install -q -U transformers accelerate qwen-vl-utils datasets scikit-learn

from huggingface_hub import login
try:                                    # Colab
    from google.colab import userdata
    login(userdata.get("HF_TOKEN"))
except ImportError:                     # Kaggle
    from kaggle_secrets import UserSecretsClient
    login(UserSecretsClient().get_secret("HF_TOKEN"))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 57.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 64.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 39.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.


In [2]:
import os, re, math, json
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from datasets import load_dataset
from sklearn.metrics import classification_report, confusion_matrix

MODEL_ID = "google/gemma-3-4b-it"
TAG = "gemma3_4b"                        # used in output file names
SEED = 42
MAX_NEW_TOKENS = 16                       # answer is one letter or yes/no
OUT_CSV = f"vqa_results_{TAG}.csv"
SAMPLE_CSV = "vqa_sample.csv"             # same file for all four models
torch.manual_seed(SEED)

## 2. Load the data (closed + multiple_choice only)

In [3]:
ds = load_dataset("raidium/RadImageNet-VQA", "benchmark", split="test")

meta = ds.remove_columns("image").to_pandas()
meta["content_type"] = meta["metadata"].apply(lambda m: m["content_type"])
meta["anatomy"]      = meta["metadata"].apply(lambda m: m["location"])
meta["question_id"]  = meta["metadata"].apply(lambda m: m["question_id"])
meta["idx"] = np.arange(len(meta))        # row position in ds, used to fetch the image later

GROUPS = {   # (question_type, content_type) -> group name
    ("closed", "anatomy"):             "anatomy_yesno",
    ("closed", "pathology"):           "abnormality_yesno",   # "Does the X look abnormal?"
    ("closed", "pathology_specific"):  "pathology_yesno",
    ("multiple_choice", "anatomy"):    "anatomy_mcq",
    ("multiple_choice", "pathology"):  "pathology_mcq",
}
meta["group"] = [GROUPS.get((q, c)) for q, c in zip(meta.question_type, meta.content_type)]
pool = meta[meta.group.notna()].copy()    # drops the 2,000 open-ended questions
pool["answer"] = pool["answer"].str.strip()
pool.loc[pool.question_type == "closed", "answer"] = pool["answer"].str.lower()

print(len(pool), "questions in the pool")
print(pool.groupby("group").size())

README.md:   0%|          | 0.00/13.9k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/59 [00:00<?, ?it/s]

benchmark/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  361MB            

benchmark/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/9000 [00:00<?, ? examples/s]

7000 questions in the pool
group
abnormality_yesno    1000
anatomy_mcq          1000
anatomy_yesno        2000
pathology_mcq        1000
pathology_yesno      2000
dtype: int64


## 3. Stratified sample (seed 42)
100 per group; each yes/no group is 50 `yes` + 50 `no`. The sample IDs are saved to a CSV (IDs only), and the same seed gives all four models identical questions.

In [4]:
closed = pool[pool.question_type == "closed"]
mcq    = pool[pool.question_type == "multiple_choice"]

parts = [g.groupby("answer").sample(50, random_state=SEED) for _, g in closed.groupby("group")]
parts += [g.sample(100, random_state=SEED) for _, g in mcq.groupby("group")]
sample = pd.concat(parts).sort_values("idx").reset_index(drop=True)

# smoke test: 1 question per group, never in the evaluation sample
smoke = pool.drop(index=pool.index[pool.idx.isin(sample.idx)]).groupby("group").sample(1, random_state=SEED)

# IDs only (no question text or answers): the dataset licence forbids redistribution
sample[["idx", "question_id", "group"]].to_csv(SAMPLE_CSV, index=False)

print(len(sample), "evaluation questions;", len(smoke), "smoke-test questions")
print(pd.crosstab(sample.group, sample.answer.where(sample.question_type == "closed", "MCQ")))

500 evaluation questions; 5 smoke-test questions
answer             MCQ  no  yes
group                          
abnormality_yesno    0  50   50
anatomy_mcq        100   0    0
anatomy_yesno        0  50   50
pathology_mcq      100   0    0
pathology_yesno      0  50   50


## 4. Load the model

In [5]:
from transformers import AutoModelForImageTextToText, AutoProcessor

model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto")
processor = AutoProcessor.from_pretrained(MODEL_ID)
model.eval()
print(model.device)

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

cuda:0


## 5. Prompts, inference and parsing

In [6]:
LETTERS = "ABCD"

def build_prompt(row):
    if row["question_type"] == "multiple_choice":
        opts = "\n".join(f"{L}. {c}" for L, c in zip(LETTERS, row["choices"]))
        return ("You are a medical assistant. Look at this image and answer the following multiple-choice "
                "question with ONLY the letter of the correct option (e.g. 'B'), nothing else.\n\n"
                f"Question: {row['question']}\nOptions:\n{opts}")
    return ("You are a medical assistant. Look at this image and answer the following question "
            "with ONLY the single word 'Yes' or 'No'.\n\n"
            f"Question: {row['question']}")

def predict(image, prompt, max_new_tokens=MAX_NEW_TOKENS):
    messages = [{"role": "user", "content": [{"type": "image", "image": image},
                                             {"type": "text", "text": prompt}]}]
    inputs = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=True,
                                           return_dict=True, return_tensors="pt").to(model.device, dtype=torch.bfloat16)
    input_len = inputs["input_ids"].shape[-1]
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return processor.decode(out[0][input_len:], skip_special_tokens=True).strip()

def parse_mcq(output):
    m = re.search(r"\b([A-D])\b", output)          # first standalone option letter
    return m.group(1) if m else "unparsed"

def parse_yesno(output):
    m = re.search(r"\b(yes|no)\b", output.lower())  # whichever appears first
    return m.group(1) if m else "unparsed"

def parse(row, output):
    m = re.search(r"<answer>(.*?)</answer>", output, re.S | re.I)   # MedVLM-R1 may answer as <answer>X</answer>
    text = m.group(1) if m else output
    return parse_mcq(text) if row["question_type"] == "multiple_choice" else parse_yesno(text)

def run_one(row):
    image = ds[int(row["idx"])]["image"].convert("RGB")
    raw = predict(image, build_prompt(row))
    return raw, parse(row, raw)

## 6. Smoke test (5 questions, not in the results)
Check that outputs parse, nothing runs out of memory, and the time per question.

In [7]:
import time
for _, r in smoke.iterrows():
    t = time.time()
    raw, pred = run_one(r)
    print(f"[{r.group}] true={r.answer!r} pred={pred!r} raw={raw!r} ({time.time()-t:.1f}s)")

[abnormality_yesno] true='yes' pred='yes' raw='Yes' (5.5s)
[anatomy_mcq] true='C' pred='C' raw='C' (3.8s)
[anatomy_yesno] true='yes' pred='yes' raw='Yes' (3.7s)
[pathology_mcq] true='C' pred='B' raw='B' (3.8s)
[pathology_yesno] true='no' pred='yes' raw='Yes' (3.7s)


## 7. Run the 500 questions (resumes if the session restarts)

In [8]:
done = pd.read_csv(OUT_CSV) if os.path.exists(OUT_CSV) else pd.DataFrame()
rows = done.to_dict("records")
seen = set(done["idx"]) if len(done) else set()

for _, r in tqdm(sample.iterrows(), total=len(sample)):
    if r["idx"] in seen:
        continue
    raw, pred = run_one(r)
    rows.append({"idx": r["idx"], "question_id": r["question_id"], "group": r["group"],
                 "anatomy": r["anatomy"], "question": r["question"],
                 "true_answer": r["answer"], "predicted_answer": pred, "raw_output": raw})
    if len(rows) % 50 == 0:
        pd.DataFrame(rows).to_csv(OUT_CSV, index=False)

results = pd.DataFrame(rows)
results.to_csv(OUT_CSV, index=False)
print(len(results), "rows saved to", OUT_CSV)

  0%|          | 0/500 [00:00<?, ?it/s]

500 rows saved to vqa_results_gemma3_4b.csv


## 8. Score
Unparsed outputs count as wrong and stay in the denominator.

In [9]:
def wilson(k, n, z=1.96):
    if n == 0: return (0, 0)
    p = k / n; d = 1 + z*z/n
    c = (p + z*z/(2*n)) / d
    h = z * math.sqrt(p*(1-p)/n + z*z/(4*n*n)) / d
    return c - h, c + h

results["correct"] = results.true_answer == results.predicted_answer
n, k = len(results), int(results.correct.sum())
lo, hi = wilson(k, n)
print(f"Overall accuracy: {k}/{n} = {k/n:.3f}  (95% CI {lo:.3f}-{hi:.3f})")
print(f"Parsed cleanly:   {(results.predicted_answer != 'unparsed').sum()}/{n}\n")

by_group = results.groupby("group").correct.agg(["sum", "count", "mean"]).rename(columns={"sum": "correct", "count": "n", "mean": "accuracy"})
print(by_group.round(3), "\n")
print(results.groupby("anatomy").correct.agg(["count", "mean"]).round(3).rename(columns={"count": "n", "mean": "accuracy"}), "\n")

yn = results[results.group.str.endswith("yesno")]
print(classification_report(yn.true_answer, yn.predicted_answer, labels=["yes", "no"], zero_division=0))
print(pd.DataFrame(confusion_matrix(yn.true_answer, yn.predicted_answer, labels=["yes", "no", "unparsed"]),
                   index=["true yes", "true no", "true unparsed"], columns=["pred yes", "pred no", "pred unparsed"]))

by_group.to_csv(f"vqa_scores_{TAG}.csv")

Overall accuracy: 315/500 = 0.630  (95% CI 0.587-0.671)
Parsed cleanly:   500/500

                   correct    n  accuracy
group                                    
abnormality_yesno       52  100      0.52
anatomy_mcq             84  100      0.84
anatomy_yesno           81  100      0.81
pathology_mcq           37  100      0.37
pathology_yesno         61  100      0.61 

              n  accuracy
anatomy                  
abdomen     163     0.595
ankle foot   52     0.808
brain        31     0.710
hip          38     0.526
knee         48     0.729
lung        116     0.534
shoulder     26     0.731
spine        26     0.692 

              precision    recall  f1-score   support

         yes       0.60      0.89      0.72       150
          no       0.78      0.41      0.54       150

    accuracy                           0.65       300
   macro avg       0.69      0.65      0.63       300
weighted avg       0.69      0.65      0.63       300

               pred yes  pred no

## 9. Misclassified cases

In [10]:
wrong = results[~results.correct][["group", "question", "true_answer", "predicted_answer", "raw_output"]]
print(len(wrong), "wrong")
wrong.head(30)

185 wrong


,group,question,true_answer,predicted_answer,raw_output
3,pathology_mcq,What is the most likely pathology in this image?,C,B,B
6,pathology_mcq,Select the pathology that best describes this ...,B,D,D
11,abnormality_yesno,Does the lung look abnormal?,no,yes,Yes
12,pathology_yesno,Is the abdomen affected by bowel mass?,yes,no,No
17,pathology_mcq,What pathology is shown in this scan of abdomen,A,B,B
19,abnormality_yesno,Does this scan reveal any abnormalities?,no,yes,Yes
20,pathology_yesno,Is there evidence of renal lesion in this image?,no,yes,Yes
26,pathology_mcq,What pathology is shown in this scan of hip,A,C,C
33,pathology_mcq,Which of the following pathologies is present ...,A,C,C
34,anatomy_yesno,Can you identify the abdomen in this image?,yes,no,No
